In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Нейронные сети

Чтобы разобраться в нейронных сетях и глубоком обучении, нужно идти от простого перцепторна к нейронным сетям. Посмотрим на перцептрон:

 <img src="./images/perceptron.png" alt="Тензор" title="Тензор" width="300" />

Что здесь происходит? Слева находятся входы ( $X_1, X_2, X_3$ ). В контексте анализа данных о книгах это могут быть количество страниц, год издания или цена.

Каждый вход имеет свой вес ( $W_1, W_2, W_3$ ). Вес определяет «важность» конкретного признака для нейрона.

Сверху подведено смещение ( $w_0$ или $b$ ). Оно позволяет нейрону гибко настраивать порог срабатывания, даже если все входные иксы равны нулю.

В центре нейрона происходит линейная операция — вычисление взвешенной суммы:

$$z = w_0 + \sum_{j=1}^{p} w_{j} X_j$$

Это «сырой» результат работы нейрона. На схеме он обозначен как Threshold Sum. Здесь признаки объединяются в одно число, которое еще не является финальным ответом.

Дальше идет Функция активации (Step Threshold). Это «пороговый» механизм. В оригинальном перцептроне использовалась жесткая ступенчатая функция. Если сумма $z$ больше нуля, на выходе получаем 1 (нейрон «активен»). Если сумма меньше или равна нулю, на выходе 0.

Внизу показан процесс обучения. Спрогнозированный выход сравнивается с реальным значением ( $y$ ).

Разница между ними — это Ошибка (Error).

На основании этой ошибки происходит Обновление весов ( $\Delta W$ ). Алгоритм корректирует $W$ так, чтобы в следующий раз при таких же входах ошибка стала меньше.

Теперь эту абстрактную модель перцептрона нужно контрезировать так: 

$$f(X) = \beta_0 + \sum_{k=1}^{K} \beta_k g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)$$

Одиночный нейрон в этой формуле вот:

$$g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)$$

Иксы - переменные, каждая из которых умножается на свой вес, потом происходит суммирование, результат которого переходит в функцию активации. Таких нейронов может быть много. Если их собрать в сеть, то мы как раз получим:

$$\sum_{k=1}^{K} g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)$$

А вот это уже можно поместить в линейную форму и начать обучаться, то есть как раз перейти к полной форме для всех нейронов:

$$f(X) = \beta_0 + \sum_{k=1}^{K} \beta_k g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)$$

Это можно для удобства переписать и так:

Мы уже видели, что методы машинного обучения крутятся вокруг суммы. Нейронные сети от этого уходят недалеко:

$$

\begin{aligned}
f(X) &= \beta_0 + \sum_{k=1}^{K} \beta_k h_k(X) \\
&= \beta_0 + \sum_{k=1}^{K} \beta_k g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)
\end{aligned}

$$

Как образовалась эта формула мы видели. Здесь $h_k(X)$ — скрытый нейрон (Hidden Unit).

Если бы мы не трансформировали $h_k(X)$, то у нас получилась бы обычная линейная регрессия. Однако мы применяем какую-то нелиненую трансформацию: сигмоида $\sigma(z)$ или ReLU $(z)$. Итак, еще раз посмотрим на это:

$$ g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right) $$

В скобках "сырой" нейрон. Результат, который мы получим в скобках, мы пропустим через функцию g, которая как раз и может быть, например, ReLU.

Итак, здесь $w_{k0}$ — смещение нейрона (Neuron Bias). Это можно считать "пороговой" функцией, тем значением, с которого нейрон "срабатывает". 

$\sum_{j=1}^{p}$ — сумматор входов. Входами считаются здесь переменные, то есть x. 

$w_{kj}$ — входные веса (Input Weights). Параметры матрицы $W$. Они показывают, как сильно $j$-й признак (например, num_pages) влияет на активацию $k$-го нейрона. $X_j$ — Входной признак. Конкретное значение $j$-го признака (например, "350 страниц").

Вся формула описывает двухэтапный процесс. Внутри ( $w, X$ ) мы берем признаки $X$, взвешиваем их весами $w$ и пропускаем через нелинейность $g$. Получаем новые, "умные", признаки $h(X)$. Снаружи ( $\beta, h$ ): мы берем эти новые признаки $h(X)$ и строим над ними обычную линейную комбинацию с весами $\beta$.

Отличие от обычной линейной модели в том, что у нас бета умножается не просто на исходное X, а на некое преобразованное значение, построенное на основе X. Как мы уже говорили, машинное обучение не может вырваться за пределы суммирования, а значит ему нужно как-то изменять исходные данные. Нейронная сеть и есть такой способ изменения. Функция g здесь становится основой всего. Популярной является функция ReLU:

$$

g(z) = (z)_+ = \begin{cases} 
0, & \text{если } z < 0 \\ 
z & \text{в противном случае} 
\end{cases}

$$

Предположим, что мы внтури g получили результат -5.4. Подставляем в ReLU. Так как $-5.4 \le 0$, то $g(-5.4) = 0$. 

Давайте посмотрим, как работает эта наша формула.

Берем уже известную формулу:
$$f(X) = \beta_0 + \sum_{k=1}^{K} \beta_k g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right)$$

Нам дано $K=2$ нейрона, $p=2$ входа. Кроме того, мы знаем веса и беты:

$$

\begin{aligned}
\text{Нейрон 1:} \quad & w_{10} = 0, & w_{11} = 1, & w_{12} = 1 \\
\text{Нейрон 2:} \quad & w_{20} = 0, & w_{21} = 1, & w_{22} = -1
\end{aligned}

$$

$$\beta_0 = 0, \quad \beta_1 = \frac{1}{4}, \quad \beta_2 = -\frac{1}{4}$$

Вычисляем скрытый слой $h_k$.

Сначала формируется линейная комбинация входов $z$, затем применяется функция активации $g(z) = z^2$.

**Нейрон 1 (Сумма)**
Вычисляем взвешенную сумму:
$$z_1 = w_{10} + w_{11}X_1 + w_{12}X_2 = 0 + 1\cdot X_1 + 1\cdot X_2 = X_1 + X_2$$
Применяем квадратичную активацию:
$$h_1(X) = (z_1)^2 = (X_1 + X_2)^2$$

**Нейрон 2 (Разность)**
Вычисляем взвешенную сумму:
$$z_2 = w_{20} + w_{21}X_1 + w_{22}X_2 = 0 + 1\cdot X_1 + (-1)\cdot X_2 = X_1 - X_2$$
Применяем квадратичную активацию:
$$h_2(X) = (z_2)^2 = (X_1 - X_2)^2$$

Теперь вычисляем выходной слой ( $f(X)$ ). Подставляем полученные значения $h_1$ и $h_2$ в линейное уравнение выходного слоя:

$$f(X) = \beta_0 + \beta_1 h_1(X) + \beta_2 h_2(X)$$

Делаем подстановку числовых значений весов $\beta$:
$$f(X) = 0 + \frac{1}{4}(X_1 + X_2)^2 + \left(-\frac{1}{4}\right)(X_1 - X_2)^2$$

Выносим общий множитель $\frac{1}{4}$ за скобки:
$$f(X) = \frac{1}{4} \left[ (X_1 + X_2)^2 - (X_1 - X_2)^2 \right]$$

Чтобы убедиться, что результатом действительно является умножение, раскроем скобки по формулам сокращенного умножения:
1.  Квадрат суммы: $(a+b)^2 = a^2 + 2ab + b^2$
2.  Квадрат разности: $(a-b)^2 = a^2 - 2ab + b^2$

Подставляем эти раскрытия в наше уравнение:
$$f(X) = \frac{1}{4} \left[ (X_1^2 + 2X_1X_2 + X_2^2) - (X_1^2 - 2X_1X_2 + X_2^2) \right]$$

Раскрываем внутренние скобки, меняя знаки у вычитаемого выражения:
$$f(X) = \frac{1}{4} \left[ X_1^2 + 2X_1X_2 + X_2^2 - X_1^2 + 2X_1X_2 - X_2^2 \right]$$

Группируем подобные члены. Квадраты переменных с противоположными знаками взаимно уничтожаются:

$$

\begin{aligned}
X_1^2 - X_1^2 &= 0 \\
X_2^2 - X_2^2 &= 0 \\
2X_1X_2 + 2X_1X_2 &= 4X_1X_2
\end{aligned}

$$

В результате в скобках остается только удвоенное перекрестное произведение:
$$f(X) = \frac{1}{4} [ 4 X_1 X_2 ]$$

Сокращаем четверки:
$$f(X) = X_1 X_2$$

Вот как это можно представить графически. Нужно помнить, что 

$$ A_k = h_k(X) = g \left( w_{k0} + \sum_{j=1}^{p} w_{kj} X_j \right), $$

<img src="./images/neur.png" alt="Тензор" title="Тензор" width="300" />

У нас есть формула, которая собирает результаты срабатывания нейронов. Где же здесь обучение? Обучением считается изменение весов. Не сами веса, а именно возможность их изменения. Мы изменяем веса таким образом, чтобы наша формула давала нам правдивые зависимые переменные. Давайте посмотрим, какая математика за этим. 

Сначала нам надо определить функцию потерь, то есть такую функцию, которая устанавливает, насколько предсказанное значение хуже реального. Это и есть математический анализ идеи "обучить так, чтобы предсказывала лучше":

$$E = \frac{1}{2} (y - f(X))^2$$

$f(X)$, если посмотреть чуть выше, зависит от бет и иксов. Наша цель — минимизировать $E$, значит нужна производная от $f(X)$, но эту производную нельзя посчитать без производной по бете, потому что изменение $f(X)$ зависит от изменения беты.

В итоге имеем:

$$\frac{\partial E}{\partial \beta_k} = \frac{\partial E}{\partial f} \cdot \frac{\partial f}{\partial \beta_k}$$

Мы определили функцию потерь как половину квадрата разности между истиной ( $y$ ) и предсказанием ( $f$ ):

$$E = \frac{1}{2}(y - f)^2$$

Чтобы найти производную по $f$, мы используем правило дифференцирования сложной функции: $(u^n)' = n \cdot u^{n-1} \cdot u'$.
Пусть $u = (y - f)$. Тогда:

Внешняя производная (степенная): $2 \cdot \frac{1}{2} \cdot (y - f)^{2-1} = (y - f)$.

Внутренняя производная (по переменной $f$ ): $\frac{\partial}{\partial f}(y - f) = -1$ (так как $y$ для нас — константа, а производная $-f$ равна $-1$ ).

Перемножаем: $(y - f) \cdot (-1) = -(y - f)$.

Знак минуса здесь критически важен: он разворачивает направление градиента в сторону уменьшения ошибки.

Теперь ищем $\frac{\partial f}{\partial \beta_k}$.

Мы ищем частную производную $\frac{\partial f}{\partial \beta_k}$. Это значит, что все остальные слагаемые мы считаем константами:

$\beta_0$ — константа, производная равна $0$;

$\beta_i h_i(X)$ (где $i \neq k$ ) — константы, их производные равны $0$;

Остается только слагаемое $\beta_k h_k(X)$.

Так как $h_k(X)$ в данном контексте — это просто коэффициент при переменной $\beta_k$, производная берется как $(c \cdot x)' = c$:

$$\frac{\partial}{\partial \beta_k} (\beta_k h_k(X)) = h_k(X)$$

Можно переписать 

$$\frac{\partial E}{\partial \beta_k} = \frac{\partial E}{\partial f} \cdot \frac{\partial f}{\partial \beta_k}$$

 как 

$$\Delta \beta_k = -\eta \frac{\partial E}{\partial \beta_k}$$

Тогда после преобразований выше:

$$\Delta \beta_k = \eta (y - f(X)) h_k(X)$$

Так мы определили градиентный спуск для бет. Остаются веса. Как обновить веса $w_{kj}$, которые спрятаны внутри нелинейности $g$?

$$\frac{\partial E}{\partial w_{kj}} = \underbrace{\frac{\partial E}{\partial f} \cdot \frac{\partial f}{\partial h_k}}_{\text{ошибка с выхода}} \cdot \underbrace{\frac{\partial h_k}{\partial z_k}}_{\text{производная активации}} \cdot \underbrace{\frac{\partial z_k}{\partial w_{kj}}}_{\text{вход}}$$

Где $z_k = w_{k0} + \sum w_{kj} X_j$. Разберем компоненты:

$\frac{\partial E}{\partial f} = -(y - f(X))$;

$\frac{\partial f}{\partial h_k} = \beta_k$ (ошибка распределяется пропорционально весу связи);

$\frac{\partial h_k}{\partial z_k} = g'(z_k)$ — производная функции активации (ReLU или сигмоиды);

$\frac{\partial z_k}{\partial w_{kj}} = X_j$.

Будем брать производные по элементам.

( $\frac{\partial E}{\partial f}$ ): Производная квадратичной ошибки по выходу:

$$\frac{\partial}{\partial f} \left[ \frac{1}{2}(y - f)^2 \right] = -(y - f)$$

( $\frac{\partial f}{\partial h_k}$ ): Насколько выход $f$ изменится при изменении $h_k$. Поскольку $f = \dots + \beta_k h_k + \dots$, производная равна весу связи:

$$\frac{\partial f}{\partial h_k} = \beta_k$$

( $\frac{\partial h_k}{\partial z_k}$ ): Прохождение сигнала через нелинейную функцию активации $g$:

$$\frac{\partial}{\partial z_k} [g(z_k)] = g'(z_k)$$

( $\frac{\partial z_k}{\partial w_{kj}}$ ): Влияние конкретного веса на сумму входов $z_k = \dots + w_{kj} X_j + \dots$:

$$\frac{\partial z_k}{\partial w_{kj}} = X_j$$

Теперь подставляем все звенья в формулу шага $\Delta w_{kj} = -\eta (\dots)$:

$$\Delta w_{kj} = -\eta \cdot \left[ \underbrace{-(y - f)}_{\text{Звено 1}} \cdot \underbrace{\beta_k}_{\text{Звено 2}} \cdot \underbrace{g'(z_k)}_{\text{Звено 3}} \cdot \underbrace{X_j}_{\text{Звено 4}} \right]$$

Минус перед $\eta$ и минус из производной ошибки сокращаются, давая итоговое выражение:

$$\Delta w_{kj} = \eta \cdot \underbrace{(y - f(X)) \beta_k}_{\text{проброшенная ошибка}} \cdot \underbrace{g'(z_k)}_{\text{фильтр}} \cdot \underbrace{X_j}_{\text{вход}}$$

Если в качестве $g$ вы используете ReLU, то её производная $g'(z)$ крайне проста, что делает вычисления очень быстрыми:

$$g'(z) = \begin{cases} 1, & z > 0 \\ 0, & z \le 0 \end{cases}$$

Это означает, что если нейрон был "выключен" (выдал 0 на прямом проходе), то градиент через него не течет ( $\Delta w = 0$ ), и он не обучается на этом шаге.

Процесс обновления весов после вычисления градиента (шага $\Delta w$ ) выглядит так:

$$w_{kj}^{(new)} = w_{kj}^{(old)} + \Delta w_{kj}$$

Предположим, мы предсказываем рейтинг книги (от 0 до 5).

Исходные данные и состояние сети:

Вход ( $X_j$ ): Количество страниц = $300$.

Истинный рейтинг ( $y$ ): $5.0$.

Текущее предсказание сети ( $f(X)$ ): $3.0$.

Вес внешнего слоя ( $\beta_k$ ): $0.5$.

Текущий вес внутреннего слоя ( $w_{kj}$ ): $0.01$.

Скорость обучения ( $\eta$ ): $0.0001$.

Производная активации ( $g'$ ): Пусть нейрон активен (ReLU), тогда $g' = 1$.

Вычисляем компоненты $\Delta w_{kj}$:

Ошибка: $(y - f(X)) = 5.0 - 3.0 = 2.0$. (Сеть занизила рейтинг).

Проброшенная ошибка: $2.0 \cdot 0.5 = 1.0$.

Итоговый шаг ( $\Delta w_{kj}$ ):

$$\Delta w_{kj} = 0.0001 \cdot 1.0 \cdot 1 \cdot 300 = 0.03$$

Обновление веса:

$$w_{kj}^{(new)} = 0.01 + 0.03 = 0.04$$

Вес увеличился. В следующий раз при входе «300 страниц» этот нейрон выдаст более высокое значение, и итоговое предсказание $f(X)$ станет ближе к $5.0$.

Остается сделать несколько замечаний перед тем, как мы попробуем запустить нейронную сеть. 

1. Персептрон очень напоминает логистическую регрессию. Это хорошо видно на следующем рисунке

<img src="./images/tensor3.png" alt="Тензор" title="Тензор" width="300" />

2. Этапы применения нейронной сети следующие:

- сначала задаем обработку данных и слои нейронной сети;

- затем компилируем, задаем способ оптимизации, метрики;

- обучаем;

- делаем предсказания.

3. Важнейшие понятия машинного обучения и нейронных сетей.

- Инициализация модели, параметров. Это задание функции, начальных значений весов для независимыех переменных, смещения. Как пример, задание весов и смещения в линейной регрессии. В случае, если используется вероятностный подход (например, логистическая регрессия), формула линейной регрессии дополнительно "помещается" в функцию, которая расчитывает вероятность, например softmax.

- Функция активации. В каждом слое применяется линейное преобразование, например функция линейной регрессии. Эта функция "оборачивается" нелинейной функцией. Например, функцией активации, в которую "помещается" линейная функция, будет ReLU, сигмоидная, тангенциальная функция. Задача указанных функций в том, чтобы на них можно было с большей эффективностью вычислять минимумы.

- Функция потерь. Это функция, которая измеряет разницу между фактическим значением зависимой переменной и значением зависимой переменной, которое было спрогнозировано моделью машинного обучения. В обычной линейной регрессии функцией потерь может быть среднеквадратичная ошибка, в логистической регрессии функция потерь будет основана на методе максимального правдоподобия. Здесь может быть применена регуляризаця, когда к значению потери прибавляется дополнительный (штрафной) член, а задача минимизации значения потери становится задачей минимизации потери плюс штрафа.

- Методы оптимизации (градиентный спуск и т.д.). Это способ нахождения минимума функции потерь за счет обновления параметров модели.

- Обучение модели. Это процесс, который начинается с инициализации параметров, продолжается прогнозированием значения зависимой переменной, расчетом функции потерь, а затем применением выбранного метода оптимизации. 

In [2]:
data = pd.read_csv(os.getcwd() + '\\gd_clean_data.csv') 
db = data.copy()
db = db.drop(['title', 'language_code', 'authors', 'editions_count', 'year', 'quarter'], axis=1)
db.head(3)

,bookID,average_rating,num_pages,ratings_count,text_reviews_count
0,41865,3.59,501,4597666,94265
1,5907,4.27,366,2530894,32871
2,5107,3.80,277,2457092,43499


In [3]:
# 1. Подготовка данных (используем ваш db)
X = db.drop('average_rating', axis=1)
y = db['average_rating']

# 2. Разделение на обучение и тест
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Стандартизация (обязательна для нейросетей)
# Приводим данные к нулевому среднему и единичной дисперсии
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Создание и обучение модели
# hidden_layer_sizes=(64, 32): два скрытых слоя с 64 и 32 нейронами
# solver='adam': стохастический градиентный оптимизатор
# early_stopping=True: остановка обучения, если ошибка перестала падать (защита от переобучения)
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    verbose=True  # Вывод хода обучения
)

print("Начинаю обучение нейросети...")
model.fit(X_train_scaled, y_train)

# 5. Предсказание и оценка
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("-" * 30)
print(f"Средняя абсолютная ошибка (MAE): {mae:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse:.4f}")
print(f"R2 Score (Коэффициент детерминации): {model.score(X_test_scaled, y_test):.4f}")

Начинаю обучение нейросети...
Iteration 1, loss = 5.24717175
Validation score: -77.273002
Iteration 2, loss = 1.63509836
Validation score: -11.254377
Iteration 3, loss = 0.41249586
Validation score: -6.144373
Iteration 4, loss = 0.28332049
Validation score: -4.125406


Iteration 5, loss = 0.20188345
Validation score: -2.740741
Iteration 6, loss = 0.14517206
Validation score: -1.751607
Iteration 7, loss = 0.10270039
Validation score: -0.976373


Iteration 8, loss = 0.07578101
Validation score: -0.546367
Iteration 9, loss = 0.06156267
Validation score: -0.279135
Iteration 10, loss = 0.05435901
Validation score: -0.153597


Iteration 11, loss = 0.05062776
Validation score: -0.092969
Iteration 12, loss = 0.04875348
Validation score: -0.058617
Iteration 13, loss = 0.04791554
Validation score: -0.068040


Iteration 14, loss = 0.04683314
Validation score: -0.037121
Iteration 15, loss = 0.04616107
Validation score: -0.024006
Iteration 16, loss = 0.04560448
Validation score: 0.003522


...

Iteration 61, loss = 0.04267774
Validation score: 0.068989
Iteration 62, loss = 0.04274982
Validation score: 0.068637
Iteration 63, loss = 0.04255989
Validation score: 0.072622


Iteration 64, loss = 0.04283975
Validation score: 0.066504
Iteration 65, loss = 0.04259465
Validation score: 0.070205
Iteration 66, loss = 0.04274494
Validation score: 0.072606


Iteration 67, loss = 0.04315806
Validation score: 0.018207
Iteration 68, loss = 0.04319898
Validation score: 0.069271
Iteration 69, loss = 0.04279778
Validation score: 0.041777


Iteration 70, loss = 0.04286816
Validation score: 0.050274
Iteration 71, loss = 0.04269577
Validation score: 0.054597


Iteration 72, loss = 0.04356833
Validation score: 0.063787
Iteration 73, loss = 0.04375924
Validation score: 0.043257
Iteration 74, loss = 0.04387045
Validation score: -0.104541
Validation score did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
------------------------------
Средняя абсолютная ошибка (MAE): 0.2187
Среднеквадратичная ошибка (MSE): 0.0872
R2 Score (Коэффициент детерминации): 0.0489


Результаты слабые, но цели обучить модель на "отлично" мы и не ставили. Наша цель - показать, как работает нейронная сеть. А это мы сделали.